In [0]:
%skip
import json

path = "/Volumes/bronze/stage/raw/Products_netsuite_schema.json"

with open(path,'w') as f:
    metadata = json.load(f)

In [0]:
%skip
#the code will work but below code is simple
import json

path = "/Volumes/bronze/stage/raw/Products_netsuite_schema.json"

with open(path, encoding="utf-8-sig") as f:
    metadata = json.load(f)

columns = metadata["schema"]
json_column = metadata["json_column"]

#display(metadata)
#display(columns)
display(json_column)

In [0]:
metadata_df = spark.read.option("multiline","true").json(
    "/Volumes/bronze/stage/raw/Products_netsuite_schema.json"
)

metadata = metadata_df.first().asDict()

columns = metadata["schema"]
json_column = metadata["json_column"]

#display(metadata)
display(columns)
#display(json_column)

In [0]:
%skip
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

fields = [
    StructField(col["name"], StringType(), True)
    for col in columns
]

schema = ArrayType(StructType(fields))
display(schema)

In [0]:
fields = []

for col in columns:
    fields.append(
        StructField(col["name"], StringType(), True)
    )
    print(col)

schema = ArrayType(StructType(fields))


display(schema)

In [0]:
df = spark.read.option("multiline","true").json(
   '/Volumes/bronze/stage/raw/Products_netsuite.json'
)

In [0]:
from pyspark.sql.functions import from_json, explode, col

df_parsed = df.withColumn(
    "parsed",
    from_json(col(json_column), schema)
)
df_exploded = df_parsed.select(
    explode("parsed").alias("item")
)
final_df = df_exploded.select("item.*")

display(final_df)